# Sentiment Classification Project

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load data

In [17]:
train_full = pd.read_csv("data/train_lang.csv")
# train_full = train_full[train_full["lang"] == "deu_Latn"]

print(train_full[5:10])
print("\n\n")
print("-" * 50)
print("\n\n")
print(train_full["sentence"].iloc[6])

   id                                           sentence  label      lang
5   5  Jürgen Boos\n\nLäuft nicht auf allen meinen Ge...      2  deu_Latn
6   6  I hope they get it togather.\n\nThe design was...      1  eng_Latn
7   7  Schön aber naja..\n\nSitzt ein bisschen weit u...      2  deu_Latn
8   8  Angenehm zu tragen.\n\nSehr praktisch. Leider ...      3  deu_Latn
9   9  Eh - for me, anyway\n\nFeel was ok-ish, didn’t...      1  eng_Latn



--------------------------------------------------



I hope they get it togather.

The design was good. The bad part it did not preform, I used an extra att. still not good on SW AIR bands. Will buy another brand.


# Build Validation Set
We use 90% of the reviews for training, and the remaining 10% for validation

In [18]:
train_df, val_df = train_test_split(
        train_full, test_size=0.1, stratify=train_full["label"], random_state=42
)

# Bag-of-words + Logistic Regression

In [19]:
from sklearn.feature_extraction.text import CountVectorizer

In [20]:
# We only keep the 10'000 most frequent words and bigrams (i.e. word pairs)
# This is both to reduce the computational cost and reduce potential overfitting
vectorizer = CountVectorizer(ngram_range=(1, 2), max_features=10000)

In [21]:
# Important: Fit ONLY on training data
X_train = vectorizer.fit_transform(train_df["sentence"])
X_val = vectorizer.transform(val_df["sentence"])

Y_train = train_df["label"]
Y_val = val_df["label"]

In [22]:
X_train

<226800x10000 sparse matrix of type '<class 'numpy.int64'>'
	with 8225774 stored elements in Compressed Sparse Row format>

In [23]:
print(X_train[2])

  (0, 7119)	3
  (0, 52)	6
  (0, 2552)	3
  (0, 5790)	1
  (0, 2342)	1
  (0, 71)	1
  (0, 7063)	1
  (0, 5368)	3
  (0, 7056)	1
  (0, 912)	1
  (0, 6421)	1
  (0, 1950)	6
  (0, 3830)	1
  (0, 5082)	1
  (0, 9084)	3
  (0, 760)	2
  (0, 2365)	1
  (0, 2277)	1
  (0, 8698)	3
  (0, 7068)	1
  (0, 2340)	1
  (0, 639)	4
  (0, 1832)	3
  (0, 3852)	1
  (0, 5982)	1
  :	:
  (0, 4242)	1
  (0, 321)	1
  (0, 7298)	1
  (0, 2348)	1
  (0, 3499)	1
  (0, 3995)	1
  (0, 2580)	1
  (0, 4261)	1
  (0, 8795)	1
  (0, 9100)	1
  (0, 57)	2
  (0, 649)	1
  (0, 9875)	1
  (0, 908)	1
  (0, 5352)	1
  (0, 5751)	1
  (0, 8764)	1
  (0, 5370)	1
  (0, 726)	1
  (0, 4252)	1
  (0, 1732)	1
  (0, 2056)	1
  (0, 4546)	1
  (0, 641)	1
  (0, 308)	1


Now we train a logistic regression classifier...

In [24]:
from sklearn.linear_model import LogisticRegression

In [25]:
# Logistic Regression Classifier, C: Inverse of regularization strength , max_iter: Maximum number of training iterations
model = LogisticRegression(C=1.0, max_iter=100)
model.fit(X_train, Y_train)

/home/micha/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

# Evaluate model

In [14]:
Y_train_pred = model.predict(X_train)
Y_val_pred = model.predict(X_val)

In [29]:
from sklearn.metrics import mean_absolute_error

In [28]:
# score on training set
mae_train = mean_absolute_error(Y_train, Y_train_pred)
score_train = 1.0 - (mae_train / 4.0)
accuracy_train = np.mean(Y_train == Y_train_pred)

# score on validation set
mae_val = mean_absolute_error(Y_val, Y_val_pred)
score_val = 1.0 - (mae_val / 4.0)
accuracy_val = np.mean(Y_val == Y_val_pred)

print(f"Training Score: {score_train:.4f}, MAE: {mae_train:.4f}, Accuracy: {accuracy_train:.4f}")
print(f"Validation Score: {score_val:.4f}, MAE: {mae_val:.4f}, Accuracy: {accuracy_val:.4f}")

ValueError: Found input variables with inconsistent numbers of samples: [226800, 113715]

# Make test submission

In [14]:
test_df = pd.read_csv("data/test.csv")
X_test = vectorizer.transform(test_df["sentence"])

In [16]:
submit_preds = model.predict(X_test)
submission = pd.DataFrame({
    "id": test_df["id"],
    "label": submit_preds
})

submission_path = "submissions/submission.csv"
submission.to_csv(submission_path, index=False)

# Model Interpretation

In [ ]:
# Top N most Important Words & Word Pairs per Output Class (Pos, Neutral, Negative)
feature_names = vectorizer.get_feature_names_out() # get names of all tokens from vectorizer
coefs = model.coef_  # Weights per Feature for each Output Class; Shape: (Num_Output_Classes, Num_Features)

# Get Top_n Features by Weight for each Class
def get_top_features(class_index, top_n=10):
    class_coef = coefs[class_index]
    top_indices = np.argsort(class_coef)[-top_n:]
    return [feature_names[i] for i in reversed(top_indices)]

print("Top words & bigrams for 1 stars:", get_top_features(0))
print("Top words & bigrams for 2 star:", get_top_features(1))
print("Top words & bigrams for 3 stars:", get_top_features(2))
print("Top words & bigrams for 4 stars:", get_top_features(3))
print("Top words & bigrams for 5 stars:", get_top_features(4))

In [ ]:
# Confusion Matrix - Negative, Neutral, Positive
from sklearn.metrics import confusion_matrix

conf_matrix = confusion_matrix(Y_val,Y_val_pred, labels=[0,1,2,3,4])
print(conf_matrix)

In [ ]:
results_df = val_df.copy()
results_df['predicted_label'] = Y_val_pred

results_df['diff'] = (results_df['label'] - results_df['predicted_label']).abs()
very_off = results_df[results_df['diff'] >= 3].sort_values(by='diff', ascending=False)

for i, row in very_off.head(10).iterrows():
    print(f"--- Review Snippet ---")
    print(f"{row['sentence'][:300]}...") # Increased to 300 to see more context
    print(f"Actual: {row['label']} | Predicted: {row['predicted_label']}")
    print("-" * 30 + "\n")